In [27]:
import pandas as pd
from google.colab import files

uploaded = files.upload()


print(f"File berhasil di-upload: {filename}")

df = pd.read_csv(filename)
display(df.head(3))

Saving paket_a_transaksi_pos_2024_2025 (1).csv to paket_a_transaksi_pos_2024_2025 (1) (2).csv
File berhasil di-upload: paket_a_transaksi_pos_2024_2025 (1).csv


,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status
0,TRX0241065,2025-06-02,G09,K011,RMT1402,4,13000,0,SELESAI
1,TRX0240510,2024-12-28,G09,K015,SGR1205,1,21000,0,SELESAI
2,TRX0244483,2024-09-04,G11,K029,RMT1407,3,"7500,00",0,SELESAI


In [29]:
df['harga_satuan'] = (
    df['harga_satuan']
    .astype(str)
    .str.replace(',', '.', regex=False)
    .astype(float)
)
df.loc[df['jumlah'] < 0, 'jumlah'] = df['jumlah'].abs()

df['kode_kasir'] = df['kode_kasir'].fillna('K_UNKNOWN')

task1_4_result = df.iloc[:5][['id_transaksi', 'tanggal', 'kode_gerai', 'jumlah']]
print("--- Hasil Task 1.4 ---")
display(task1_4_result)

--- Hasil Task 1.4 ---


,id_transaksi,tanggal,kode_gerai,jumlah
0,TRX0241065,2025-06-02,G09,4
1,TRX0240510,2024-12-28,G09,1
2,TRX0244483,2024-09-04,G11,3
3,TRX0242532,2025-07-30,G05,2
4,TRX0243509,26/03/2025,G05,1


In [32]:
df['Total_Kotor'] = df['harga_satuan'] * df['jumlah']

df['Nominal_Diskon'] = df['Total_Kotor'] * (df['diskon_persen'] / 100)

df['Total_Bersih'] = (df['Total_Kotor'] - df['Nominal_Diskon']) * 1.11

display(df[['id_transaksi', 'Total_Kotor', 'Nominal_Diskon', 'Total_Bersih']].head(3))

,id_transaksi,Total_Kotor,Nominal_Diskon,Total_Bersih
0,TRX0241065,52000.0,0.0,57720.0
1,TRX0240510,21000.0,0.0,23310.0
2,TRX0244483,22500.0,0.0,24975.0


In [33]:
filter_multi = df[
    (df['status'] == 'SELESAI') &
    (df['kode_gerai'].isin(['G05', 'G09'])) &
    (df['Total_Bersih'] > 100000)
]
print(f"Jumlah baris hasil Filter Multi-Kondisi: {len(filter_multi)}")

filter_query = df.query("kode_gerai == 'G10' and jumlah >= 5")
print(f"Jumlah baris hasil Filter Query (G10 & jumlah >= 5): {len(filter_query)}")

Jumlah baris hasil Filter Multi-Kondisi: 139
Jumlah baris hasil Filter Query (G10 & jumlah >= 5): 39


In [34]:
sorted_df = df.sort_values(by=['kode_gerai', 'Total_Bersih'], ascending=[True, False])

top_5 = df.nlargest(5, 'Total_Bersih')
print("--- Top 5 Transaksi Total_Bersih Tertinggi ---")
display(top_5[['id_transaksi', 'kode_gerai', 'Total_Bersih']])

--- Top 5 Transaksi Total_Bersih Tertinggi ---


,id_transaksi,kode_gerai,Total_Bersih
994,TRX0241002,G11,100363536.0
4985,TRX0244221,G09,66934498.5
467,TRX0243696,G06,53466480.0
3393,TRX0241029,G07,45563391.0
244,TRX0243257,G01,42020160.0


In [35]:
df_selesai = df[df['status'] == 'SELESAI']

laporan_gerai = df_selesai.groupby('kode_gerai').agg(
    Jumlah_Transaksi=('id_transaksi', 'count'),
    Total_Unit_Terjual=('jumlah', 'sum'),
    Total_Omzet_Bersih=('Total_Bersih', 'sum'),
    Rata_Rata_Transaksi=('Total_Bersih', 'mean')
).sort_values(by='Total_Omzet_Bersih', ascending=False).round(2)

print("--- Tabel Laporan Gerai ---")
display(laporan_gerai)

df['Kontribusi_Gerai_Persen'] = (
    df['Total_Bersih'] / df.groupby('kode_gerai')['Total_Bersih'].transform('sum')
) * 100

print("--- Kolom Kontribusi Gerai Berhasil Ditambahkan ---")
display(df[['id_transaksi', 'kode_gerai', 'Total_Bersih', 'Kontribusi_Gerai_Persen']].head(3))

--- Tabel Laporan Gerai ---


,Jumlah_Transaksi,Total_Unit_Terjual,Total_Omzet_Bersih,Rata_Rata_Transaksi
kode_gerai,,,,
G11,398,3170,1.253536e+08,314958.82
G06,383,3103,9.113936e+07,237961.77
G09,379,3027,9.019606e+07,237984.32
G07,384,4284,8.036181e+07,209275.55
G01,450,2635,7.077959e+07,157287.97
G03,372,3274,5.980970e+07,160778.77
G10,401,1742,5.892248e+07,146938.85
G08,408,1700,3.550256e+07,87016.08
G12,406,1010,2.464483e+07,60701.55


--- Kolom Kontribusi Gerai Berhasil Ditambahkan ---


,id_transaksi,kode_gerai,Total_Bersih,Kontribusi_Gerai_Persen
0,TRX0241065,G09,57720.0,0.063031
1,TRX0240510,G09,23310.0,0.025455
2,TRX0244483,G11,24975.0,0.018364


In [21]:

df_selesai = df[df['status'] == 'SELESAI']

laporan_gerai = df_selesai.groupby('kode_gerai').agg(
    Jumlah_Transaksi=('id_transaksi', 'count'),
    Total_Unit_Terjual=('jumlah', 'sum'),
    Total_Omzet_Bersih=('Total_Bersih', 'sum'),
    Rata_Rata_Transaksi=('Total_Bersih', 'mean')
).sort_values(by='Total_Omzet_Bersih', ascending=False).round(2)

print("--- Tabel Laporan Gerai ---")
display(laporan_gerai)

df['Kontribusi_Gerai_Persen'] = (
    df['Total_Bersih'] / df.groupby('kode_gerai')['Total_Bersih'].transform('sum')
) * 100

print("--- Kolom Kontribusi Gerai Berhasil Ditambahkan ---")
display(df[['id_transaksi', 'kode_gerai', 'Total_Bersih', 'Kontribusi_Gerai_Persen']].head(3))

--- Tabel Laporan Gerai ---


,Jumlah_Transaksi,Total_Unit_Terjual,Total_Omzet_Bersih,Rata_Rata_Transaksi
kode_gerai,,,,
G11,398,3170,1.253536e+08,314958.82
G06,383,3103,9.113936e+07,237961.77
G09,379,3027,9.019606e+07,237984.32
G07,384,4284,8.036181e+07,209275.55
G01,450,2635,7.077959e+07,157287.97
G03,372,3274,5.980970e+07,160778.77
G10,401,1742,5.892248e+07,146938.85
G08,408,1700,3.550256e+07,87016.08
G12,406,1010,2.464483e+07,60701.55


--- Kolom Kontribusi Gerai Berhasil Ditambahkan ---


,id_transaksi,kode_gerai,Total_Bersih,Kontribusi_Gerai_Persen
0,TRX0241065,G09,57720.0,0.063031
1,TRX0240510,G09,23310.0,0.025455
2,TRX0244483,G11,24975.0,0.018364
